# Which hosted Nemotron model for which workload? Reasoning on or off?

Question: which hosted Nemotron model should serve which workload, and when is reasoning worth its extra tokens and latency?

Setup: a CPU notebook that calls the NVIDIA-hosted, OpenAI-compatible API (`https://integrate.api.nvidia.com/v1`) with Python standard library plus pandas. It needs an `NVIDIA_API_KEY` secret. 30 tasks with exact answers (10 arithmetic word problems, 10 JSON field extractions, 10 message routings) run on 5 hosted models. Models that reason run twice: with their default reasoning and with reasoning turned off.

Terms used below:

- **Token**: the unit of text a model reads and writes. One token is often a word piece, not a whole word.
- **Autoregressive generation**: the model writes one token per step. Each new token depends on the tokens before it.
- **Reasoning model**: a model that writes a long chain of thought (thinking tokens) before its final answer. The thinking tokens count as output tokens.
- **Dense model**: every weight is used for every token.
- **Mixture of experts (MoE)**: the model holds many expert sub-networks. A router picks a few for each token. The name `30b-a3b` means 30B total parameters and 3B active per token.
- **Small language model (SLM)**: a model with a few billion parameters or fewer. It is cheap and fast, but it may be less accurate.
- **p95 latency**: 95 out of 100 requests finish at or below this time.
- **Compute proxy**: completion tokens times active parameters in billions. The hosted endpoint shows no price, so this number only ranks models by compute work. It is not a price.
- **NIM**: NVIDIA Inference Microservice. A packaged model server with an OpenAI-compatible API. This notebook calls NVIDIA-hosted NIM endpoints.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Question | What you measure | Your guess |
|---|---|---|
| Does a bigger model score higher on each workload? | Accuracy per model and workload | |
| Does the smallest model meet the floor on any workload? | Accuracy of the 4B and 3B-active models | |
| Which workload gains most from reasoning? | Accuracy, reasoning on against off | |
| How many more tokens does reasoning use? | Completion tokens, default against off | |
| Can a model run out of tokens before it answers? | Count of empty replies with `finish_reason=length` | |
| Which model wins on proxy cost for each workload? | Cheapest model above the floor and under the p95 limit | |
| Is a router cheaper than always using the largest model? | Accuracy, tokens and proxy, router against largest | |
| How does latency change with output length? | Slope of latency against completion tokens | |

In [ ]:
# 1. Setup: constants and API key (nothing to install: standard library + pandas)
import http.client, json, os, random, re, statistics, time, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

BASE = os.environ.get("NIM_BASE_URL", "https://integrate.api.nvidia.com/v1")
MODELS = {  # model id -> active parameters in billions (input of the compute proxy)
    "nvidia/nemotron-3.5-bolt-e4b": 4,               # treated as 4B
    "nvidia/nemotron-3.5-lightning-30b-a3b": 3,      # MoE: 3B active of 30B
    "nvidia/nemotron-3.5-super-vl-120b-a12b": 12,    # MoE: 12B active of 120B
    "nvidia/nemotron-3-ultra-550b-a55b": 55,         # MoE: 55B active of 550B
    "google/gemma-3-4b-it": 4,                       # dense 4B, non-Nemotron baseline
}
LARGEST = "nvidia/nemotron-3-ultra-550b-a55b"
SEED, MAX_TOKENS = 42, 2048          # 2048: a reasoning model needs room to think and then answer
REQ_TIMEOUT_S, MAX_RETRIES, BACKOFF_S, RETRY_CAP_S = 90, 4, 2.0, 30.0
CONCURRENCY = 2
PROBE_BUDGET_S, CONFIG_BUDGET_S, TOTAL_BUDGET_S = 180, 120, 20 * 60
ACC_FLOOR, P95_LIMIT_S, MIN_N = 0.8, 30.0, 4   # selection rule on 5 tasks: accuracy >= floor, p95 <= limit, at least MIN_N scored tasks
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def get_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("NVIDIA_API_KEY")
    except Exception:
        key = os.environ.get("NVIDIA_API_KEY")
    if not key:
        raise RuntimeError("NVIDIA_API_KEY not found. Add-ons > Secrets > add NVIDIA_API_KEY, then switch it on for this notebook.")
    return key

KEY = get_key()
print("key found:", bool(KEY))   # never print the key itself

In [ ]:
# 2. Helpers: one chat call with timeout, retry and backoff; parallel map with a deadline
def msgs(prompt, system=None):
    return ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]

def chat(model, messages, extra=None, max_tokens=MAX_TOKENS, deadline=None):
    body = {"model": model, "messages": messages, "max_tokens": max_tokens,
            "temperature": 0.6, "top_p": 0.95, "seed": SEED, **(extra or {})}
    data = json.dumps(body).encode()
    out = dict(ok=False, error="", content="", reasoning="", finish="", prompt_tokens=0,
               completion_tokens=0, reasoning_tokens=0, latency_s=float("nan"), retries=0)
    for attempt in range(MAX_RETRIES + 1):
        timeout = REQ_TIMEOUT_S if deadline is None else max(1, min(REQ_TIMEOUT_S, deadline - time.time()))
        req = urllib.request.Request(BASE + "/chat/completions", data,
                                     {"Authorization": "Bearer " + KEY, "Content-Type": "application/json"})
        t0, wait = time.time(), None
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                j = json.loads(r.read())
            ch = j["choices"][0]; m = ch["message"]; u = j.get("usage") or {}
            out.update(ok=True, error="", content=m.get("content") or "",
                       reasoning=m.get("reasoning_content") or m.get("reasoning") or "",
                       finish=ch.get("finish_reason") or "", prompt_tokens=u.get("prompt_tokens") or 0,
                       completion_tokens=u.get("completion_tokens") or 0,
                       reasoning_tokens=(u.get("completion_tokens_details") or {}).get("reasoning_tokens") or 0,
                       latency_s=time.time() - t0)
            return out
        except urllib.error.HTTPError as e:           # 429 and 5xx are retried, other codes are final
            out["error"] = f"HTTP {e.code}"
            if e.code != 429 and e.code < 500:
                return out
            try: wait = float(e.headers.get("Retry-After", ""))   # honour Retry-After
            except ValueError: wait = None
        except (ValueError, KeyError, IndexError, TypeError, AttributeError) as e:   # malformed reply
            out["error"] = "bad reply " + type(e).__name__
            return out
        except (OSError, http.client.HTTPException) as e:   # timeout, refused, reset, cut-off body
            out["error"] = type(getattr(e, "reason", e)).__name__
        if attempt == MAX_RETRIES:
            break
        wait = min(BACKOFF_S * 2 ** attempt if wait is None else wait, RETRY_CAP_S)
        if deadline is not None and time.time() + wait >= deadline:
            break
        time.sleep(wait); out["retries"] += 1
    return out

def pmap(fn, items, deadline):   # at most CONCURRENCY calls at once; items not started by the deadline give None
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        return list(ex.map(lambda x: fn(x) if time.time() < deadline else None, items))

def hosted_models():
    try:
        with urllib.request.urlopen(BASE + "/models", timeout=30) as r:   # works without a key
            return {m["id"] for m in json.load(r)["data"]}
    except Exception as e:
        print("could not list /models (" + type(e).__name__ + "); will try every model")
        return set(MODELS)

In [ ]:
# 3. Tasks: 30 tasks, fixed seed, exact ground truth, deterministic graders
rng = random.Random(SEED)
WORKLOADS = ["arithmetic", "extraction", "routing"]
LABELS = ["billing", "technical", "account", "shipping"]

def arithmetic(i):
    k = i % 3
    if k == 0:
        a, b, d, e = rng.randint(4, 9), rng.randint(12, 24), rng.randint(3, 6), rng.randint(5, 30)
        return (f"A bakery bakes {a} trays of rolls each day. Each tray holds {b} rolls. After {d} days, {e} rolls are unsold "
                f"and the rest are sold. How many rolls were sold?"), a * b * d - e
    if k == 1:
        a, b, c, d = rng.randint(8, 15), rng.randint(6, 12), rng.randint(3, 20), rng.randint(4, 8)
        return (f"A truck carries {a} crates. Each crate holds {b} bottles. {c} bottles break. The rest go into cartons of "
                f"{d} bottles. How many full cartons are packed?"), (a * b - c) // d
    b, c, d, e, q = rng.randint(2, 6), rng.randint(3, 9), rng.randint(10, 40), rng.randint(3, 7), rng.randint(5, 25)
    a = e * q + b * c - d
    return (f"Mia has {a} dollars. She buys {b} notebooks at {c} dollars each. Then she earns {d} dollars. She splits all her "
            f"money equally between {e} friends and keeps nothing. How many dollars does each friend get?"), q

NAMES = ["Ana Ruiz", "Ben Cole", "Chen Wei", "Dara Okoye", "Eli Brandt", "Fay Novak", "Gus Lund", "Hana Sato", "Ira Patel", "Joy Mercer"]
THINGS = ["bolts", "filters", "cables", "lamps", "hinges", "gaskets", "sensors", "valves"]
CITIES = ["Lyon", "Porto", "Graz", "Leeds", "Turin", "Gdansk", "Malaga", "Bergen"]
EXTRACT_PROMPT = ('Extract the order from the text. Reply with only a JSON object with the keys "order_id" (string), '
                  '"name" (string), "qty" (integer), "item" (string) and "city" (string).\nText: ')

def extraction(i):
    f = dict(order_id=f"{rng.choice('ABCDEF')}-{rng.randint(1000, 9999)}", name=NAMES[i % 10], qty=rng.randint(2, 480),
             item=rng.choice(THINGS), city=rng.choice(CITIES))
    if i % 2:
        text = f"Order {f['order_id']}: {f['name']} bought {f['qty']} {f['item']}. Deliver to {f['city']}."
    else:
        text = f"Hi, this is {f['name']}. Please ship {f['qty']} {f['item']} to {f['city']}. My order number is {f['order_id']}."
    return EXTRACT_PROMPT + text, f

ROUTE_TEXT = {
    "billing": ["I was charged twice for my last invoice.", "Please refund the {n} dollars taken from my card.", "My bill shows a fee I do not recognize."],
    "technical": ["The app crashes each time I open the settings page.", "Error 500 appears when I upload a file.", "The dashboard will not load on my phone."],
    "account": ["I forgot my password and cannot sign in.", "Please change the email address on my profile.", "How do I delete my account?"],
    "shipping": ["My package has not arrived after {n} days.", "The tracking page shows no update since Monday.", "The parcel went to the wrong address."],
}
ROUTE_PROMPT = "Route this customer message to one team. Teams: billing, technical, account, shipping. Reply with the team name only, in lowercase.\nMessage: "

def routing(i):
    label = LABELS[i % 4]
    return ROUTE_PROMPT + rng.choice(ROUTE_TEXT[label]).format(n=rng.randint(5, 60)), label

TASKS = []
for i in range(10):
    p, a = arithmetic(i); TASKS.append(dict(id=f"arithmetic-{i}", workload="arithmetic", prompt=p + " Reply with the final integer only.", answer=a))
    p, a = extraction(i);  TASKS.append(dict(id=f"extraction-{i}", workload="extraction", prompt=p, answer=a))
    p, a = routing(i);     TASKS.append(dict(id=f"routing-{i}", workload="routing", prompt=p, answer=a))

def grade(task, content):
    text = content.split("</think>")[-1].strip()   # some models put the thinking inside the reply
    if task["workload"] == "arithmetic":
        nums = re.findall(r"-?\d[\d,]*", text)
        return bool(nums) and nums[-1].replace(",", "") == str(task["answer"])
    if task["workload"] == "extraction":
        try:
            got = json.loads(text[text.index("{"): text.rindex("}") + 1])
        except ValueError:
            return False
        return got == task["answer"] and type(got.get("qty")) is int
    return text.lower().strip(".\"'`* \n") == task["answer"]

assert len(TASKS) == 30 and all(isinstance(t["answer"], (int, str, dict)) for t in TASKS)
print(pd.Series([t["workload"] for t in TASKS]).value_counts().to_dict())
print(TASKS[0]["prompt"], "->", TASKS[0]["answer"])

## Reasoning toggle

The hosted model pages and the NVIDIA NIM docs list three ways to change reasoning, and the right one depends on the model. This notebook does not assume which one a model supports. The next cell sends one prompt per model and per way. A way counts as working when reasoning tokens (or, if the API does not report them, completion tokens) fall by more than half against the default reply. A model with no working way runs once and is labelled `default`.

| Candidate | What it sets | Source |
|---|---|---|
| `enable_thinking=false` | `chat_template_kwargs: {"enable_thinking": false}` | Nemotron 3 NIM docs and the Bolt API page |
| `reasoning_effort=none` | top-level `reasoning_effort: "none"` | API pages of the Ultra and Super models |
| `system /no_think` | system prompt `/no_think` | NIM docs, reasoning models controlled by system prompt |

In [ ]:
# 4. Probe: which toggle changes reasoning on each model? One prompt per model and toggle.
START = time.time()
END = START + TOTAL_BUDGET_S
HOSTED = hosted_models()
USE = [m for m in MODELS if m in HOSTED]
for m in MODELS:
    if m not in HOSTED:
        print("skip (not in /v1/models):", m)

TOGGLES = {  # name -> (extra body fields, system prompt)
    "default": ({}, None),
    "enable_thinking=false": ({"chat_template_kwargs": {"enable_thinking": False}}, None),
    "reasoning_effort=none": ({"reasoning_effort": "none"}, None),
    "system /no_think": ({}, "/no_think"),
}
PROBE_PROMPT = TASKS[0]["prompt"]
jobs = [(m, t) for m in USE for t in TOGGLES]
res = pmap(lambda j: chat(j[0], msgs(PROBE_PROMPT, TOGGLES[j[1]][1]), TOGGLES[j[1]][0], max_tokens=1024,
                          deadline=START + PROBE_BUDGET_S), jobs, START + PROBE_BUDGET_S)
R = {j: r for j, r in zip(jobs, res) if r}

def thinking_size(r):
    return r["reasoning_tokens"] or len(r["reasoning"]) // 4

def works(base, r):
    if not (base["ok"] and r["ok"]):
        return False
    if thinking_size(base):
        return thinking_size(r) < 0.5 * thinking_size(base)
    return base["completion_tokens"] > 100 and r["completion_tokens"] < 0.5 * base["completion_tokens"]

TOGGLE, rows = {}, []   # TOGGLE[model] = (name, extra, system) of the first working toggle
for (m, t), r in R.items():
    w = t != "default" and (m, "default") in R and works(R[(m, "default")], r)
    if w and m not in TOGGLE:
        TOGGLE[m] = (t, *TOGGLES[t])
    rows.append(dict(model=m.split("/")[-1], toggle=t, ok=r["ok"], error=r["error"], finish=r["finish"],
                     completion_tokens=r["completion_tokens"], thinking_size=thinking_size(r),
                     latency_s=round(r["latency_s"], 1), changes_reasoning=w))
print(pd.DataFrame(rows).to_string(index=False))
print({m.split("/")[-1]: TOGGLE[m][0] if m in TOGGLE else "none found, runs as default" for m in USE})

## Runner

Each model and mode is one config. A config sends the 30 tasks in a fixed mixed order, two requests at a time, with a time budget. If the budget ends first, the config keeps the results it has and the tables show how many tasks finished (`n`). An empty reply with `finish_reason=length` is a separate outcome, `out_of_tokens`, and counts as wrong. A request that still fails after the retries is an `error`. Errors are counted but left out of accuracy.

In [ ]:
# 5. Run every config: default, and off where the probe found a working toggle
ORDER = random.Random(SEED).sample(TASKS, len(TASKS))   # mixed workloads, so a cut-off run still covers all three
ROWS = []

def run_config(model, mode, extra, system):
    deadline = min(time.time() + CONFIG_BUDGET_S, END)
    st, t0, start_rows = {"ok": 0, "err": 0}, time.time(), len(ROWS)
    def one(t):
        if st["ok"] == 0 and st["err"] >= 4:   # the model fails every call: stop early
            return
        r = chat(model, msgs(t["prompt"], system), extra, deadline=deadline)
        if not r["ok"]: outcome = "error"
        elif r["finish"] == "length" and not r["content"].strip(): outcome = "out_of_tokens"
        else: outcome = "correct" if grade(t, r["content"]) else "wrong"
        st["ok" if r["ok"] else "err"] += 1
        ROWS.append(dict(model=model, mode=mode, workload=t["workload"], task=t["id"], outcome=outcome,
                         correct=outcome == "correct", latency_s=r["latency_s"], prompt_tokens=r["prompt_tokens"],
                         completion_tokens=r["completion_tokens"], reasoning_tokens=r["reasoning_tokens"],
                         finish=r["finish"], retries=r["retries"], error=r["error"],
                         proxy=r["completion_tokens"] * MODELS[model]))
    pmap(one, ORDER, deadline)
    mine = ROWS[start_rows:]
    n_ok = sum(x["outcome"] != "error" for x in mine)
    note = "" if len(mine) == len(TASKS) else "  (cut off: " + str(len(mine)) + " of " + str(len(TASKS)) + " tasks reached)"
    print(f"{model.split('/')[-1]:36s} {mode:8s} scored {n_ok:2d}  errors {len(mine)-n_ok:2d}  "
          f"correct {sum(x['correct'] for x in mine):2d}  {time.time()-t0:5.0f}s{note}")
    if mine and n_ok == 0:
        print("   skipped: every request failed, last error:", mine[-1]["error"])

for m in sorted(USE, key=lambda m: MODELS[m]):
    run_config(m, "default", {}, None)
    if m in TOGGLE:
        run_config(m, "off", TOGGLE[m][1], TOGGLE[m][2])
print(f"total {(time.time()-START)/60:.1f} min")

In [ ]:
# 6. Summary table per model, mode and workload (also written to CSV)
RAW = pd.DataFrame(ROWS)
RAW.to_csv(f"{OUT_DIR}/nim_model_selection_raw.csv", index=False)
out = []
for (m, mode, wl), g in RAW.groupby(["model", "mode", "workload"]):
    ok = g[g.outcome != "error"]
    out.append(dict(model=m, mode=mode, workload=wl, n=len(ok), errors=len(g) - len(ok), accuracy=ok.correct.mean(),
                    out_of_tokens=int((ok.outcome == "out_of_tokens").sum()), p50_s=ok.latency_s.median(),
                    p95_s=ok.latency_s.quantile(0.95), avg_completion_tokens=ok.completion_tokens.mean(),
                    avg_proxy=ok.proxy.mean()))
SUMMARY = pd.DataFrame(out)
SUMMARY.round(2).to_csv(f"{OUT_DIR}/nim_model_selection.csv", index=False)
show = SUMMARY.assign(model=SUMMARY.model.str.split("/").str[-1]).round(2)
print(show.to_string(index=False))

In [ ]:
# 7. Is reasoning worth it? Default (reasoning as the model ships) against off, per model and workload
d = SUMMARY[SUMMARY["mode"] == "default"].set_index(["model", "workload"])
o = SUMMARY[SUMMARY["mode"] == "off"].set_index(["model", "workload"])
if o.empty:
    print("No model had a working reasoning toggle in this run, so there is no on/off pair to compare.")
else:
    dd = d.reindex(o.index)
    cmp = pd.DataFrame({"acc_default": dd.accuracy, "acc_off": o.accuracy, "acc_gain": dd.accuracy - o.accuracy,
                        "tokens_default": dd.avg_completion_tokens, "tokens_off": o.avg_completion_tokens,
                        "token_x": dd.avg_completion_tokens / o.avg_completion_tokens,
                        "p50_default_s": dd.p50_s, "p50_off_s": o.p50_s, "latency_x": dd.p50_s / o.p50_s,
                        "out_of_tokens_default": dd.out_of_tokens})
    cmp.index = cmp.index.set_levels(cmp.index.levels[0].str.split("/").str[-1], level=0)
    cmp.round(2).to_csv(f"{OUT_DIR}/nim_reasoning_on_off.csv")
    print(cmp.round(2).to_string())

In [ ]:
# 8. Pick a model per workload: lowest compute proxy that meets the accuracy floor and the p95 limit
# Pick on tasks 0 to 4 of each workload. Score the router on tasks 5 to 9, which the pick never saw.
half = RAW.task.str.split("-").str[-1].astype(int) < 5
PICK_SET, SCORE_SET = RAW[half & (RAW.outcome != "error")], RAW[~half & (RAW.outcome != "error")]
P = PICK_SET.groupby(["model", "mode", "workload"]).agg(n=("correct", "size"), accuracy=("correct", "mean"),
    p95_s=("latency_s", lambda x: x.quantile(0.95)), avg_proxy=("proxy", "mean")).reset_index()
PICK = {}
for wl in WORKLOADS:
    c = P[(P.workload == wl) & (P.n >= MIN_N) & (P.accuracy >= ACC_FLOOR) & (P.p95_s <= P95_LIMIT_S)]
    PICK[wl] = tuple(c.sort_values(["avg_proxy", "accuracy"], ascending=[True, False]).iloc[0][["model", "mode"]]) if len(c) else None
    print(f"{wl:11s} floor {ACC_FLOOR}, p95 <= {P95_LIMIT_S}s ->", PICK[wl] or "no config qualifies (router falls back to the largest model)")

def strategy(sel):   # sel: workload -> (model, mode)
    d = pd.concat([SCORE_SET[(SCORE_SET.workload == wl) & (SCORE_SET.model == m) & (SCORE_SET["mode"] == mo)] for wl, (m, mo) in sel.items()])
    return dict(tasks=len(d), accuracy=d.correct.mean(), avg_completion_tokens=d.completion_tokens.mean(),
                avg_proxy=d.proxy.mean(), avg_latency_s=d.latency_s.mean())

big = {wl: (LARGEST, "default") for wl in WORKLOADS}
if not (SCORE_SET.model == LARGEST).any():
    print("The largest model returned no scored rows, so the router cannot be compared with it.")
else:
    router = {wl: PICK[wl] or big[wl] for wl in WORKLOADS}
    cmp2 = pd.DataFrame({"router": strategy(router), "always largest": strategy(big)}).T
    cmp2.loc["router vs largest (%)"] = (cmp2.loc["router"] / cmp2.loc["always largest"] - 1) * 100
    cmp2.round(2).to_csv(f"{OUT_DIR}/nim_router_vs_largest.csv")
    print(cmp2.round(2).to_string())

## Autoregressive generation: latency against output length

The model writes one token per step, so a longer reply needs more steps. The next cell fits a line through all scored requests: latency (s) against completion tokens. The slope is seconds per output token. The intercept is the fixed cost (network, queue, prompt processing). Then it counts tokens per word for a few texts with the `prompt_tokens` number from the API (the fixed template cost is removed by subtracting a one-token baseline).

In [ ]:
# 9. models.9: latency against completion tokens, and tokens per word
ok = RAW[(RAW.outcome != "error") & (RAW.completion_tokens > 0)]
fit = []
for m, g in ok.groupby("model"):
    try:
        s, i = statistics.linear_regression(g.completion_tokens, g.latency_s)
        fit.append(dict(model=m.split("/")[-1], n=len(g), ms_per_token=1000 * s, fixed_s=i,
                        r=statistics.correlation(g.completion_tokens, g.latency_s)))
    except (statistics.StatisticsError, ValueError):
        print("not enough spread to fit:", m)
print(pd.DataFrame(fit).round(2).to_string(index=False))
bins = pd.cut(ok.completion_tokens, [0, 50, 200, 800, MAX_TOKENS + 1])
print(ok.groupby([ok.model.str.split("/").str[-1], bins], observed=True).latency_s.agg(["count", "median"]).round(2).to_string())
try:
    import matplotlib.pyplot as plt
    for m, g in ok.groupby("model"):
        plt.scatter(g.completion_tokens, g.latency_s, s=12, label=m.split("/")[-1])
    plt.xlabel("completion tokens"); plt.ylabel("latency (s)"); plt.legend(fontsize=7)
    plt.savefig(f"{OUT_DIR}/latency_vs_tokens.png", dpi=120, bbox_inches="tight"); plt.show()
except ImportError:
    print("matplotlib not available: the table above holds the same numbers")

TEXTS = {
    "english prose": "The quick brown fox jumps over the lazy dog while the sun sets behind the hills.",
    "json": '{"order_id": "A-1234", "qty": 12, "items": ["bolts", "nuts"], "paid": true}',
    "python code": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))",
    "long numbers": "3141592653 2718281828 1618033988 7499999999",
    "spanish": "El sistema procesa cada solicitud y devuelve una respuesta corta al usuario.",
}
tw = []
for m in USE[:3]:   # three models keep the cell short
    base = chat(m, msgs("a"), max_tokens=1, deadline=time.time() + 60)
    for name, text in TEXTS.items():
        r = chat(m, msgs(text), max_tokens=1, deadline=time.time() + 60)
        if base["ok"] and r["ok"]:
            n = r["prompt_tokens"] - base["prompt_tokens"] + 1
            tw.append(dict(model=m.split("/")[-1], text=name, words=len(text.split()), tokens=n, tokens_per_word=round(n / len(text.split()), 2)))
TW = pd.DataFrame(tw)
TW.to_csv(f"{OUT_DIR}/nim_tokens_per_word.csv", index=False)
print(TW.to_string(index=False) if len(TW) else "no token counts returned")

## What to look for

- **Accuracy per workload**: compare each model with the 0.8 floor. Check if a small model already clears it on routing or extraction.
- **Reasoning on against off**: read `acc_gain`, `token_x` and `latency_x` together. Reasoning is worth it only when `acc_gain` is large enough to pay for the extra tokens and seconds.
- **`out_of_tokens`**: an empty reply with `finish_reason=length` means the thinking used the whole 2048-token budget. The reply is wrong and it still costs tokens.
- **`n` and `errors` columns**: a row with fewer than 10 scored tasks was cut by the time budget or by API errors. Read its accuracy with care. Ten tasks per workload is a small sample, so a one-task change moves accuracy by 0.1.
- **Router against largest**: the router picks its models on 5 tasks per workload and is scored on the other 5. So the score is not inflated by the pick. Five tasks is a small sample, so treat the gap as a hint.
- **Proxy**: completion tokens times active parameters ranks compute work only. It ignores prompt tokens, GPU type, batching and the hosted price.
- **Latency slope**: a larger `ms_per_token` means a slower model per step. A larger `fixed_s` means queue and network time that output length does not explain.
- **Tokens per word**: code, JSON and digits often split into more tokens per word than prose. Different models use different tokenizers.

## Exam rules this proves

1. **models.7**: A mixture-of-experts name such as `30b-a3b` gives total and active parameters. Compute per token follows the active number. Memory still has to hold the total.
2. **models.9**: Autoregressive generation makes one token per step, so latency grows with the number of output tokens. Thinking tokens of a reasoning model count as output tokens.
3. **models.8, agentic.6**: Set the accuracy floor and the latency limit first. Then pick the cheapest model that meets both. Measure accuracy, tokens, latency and cost together, per workload.
4. **models.5, models.7**: Nemotron models come in several sizes, and reasoning is a request setting on some of them. Read the model card for the exact setting, and test that it changes the token count.
5. **enterprise.6**: Compute-aware selection matches model size to the workload. When no price is shown, rank models by a labelled compute proxy and say that it is a proxy.